# Exp 26: Coarse dropout, 5 folds

Card: [exp26_coarse_dropout_5fold.md](../../docs/neuroslope/experiments/exp26_coarse_dropout_5fold.md)

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import pandas as pd

from training.config import ROOT
from training.config import VIP
from training.cv import CVResult
from training.runs import read_config
from analysis.gradcam import GradCamFigures
from training.tracking import start_run
from analysis.plots import pr_curves, save_figures, use_style
from analysis.thresholds import best_f1, cross_fold_f1
from training.config import CLASSES

N_SAMPLE = 16  # exp19 sample
N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.yaml"
EXP11 = CONFIG_DIR / "exp11_resnet18_5fold" / "resnet18.yaml"
EXP16 = CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml"  # exp19 rows drawn from exp16 OOF
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp26_coarse_dropout_5fold"

# dataviz reference palette, light mode
SERIES = {"exp11": "#eb6834", "coarse dropout": "#2a78d6"}
use_style()

## Training

In [ ]:
results = {"exp11": CVResult.load(EXP11), "coarse dropout": CVResult.load(CONFIG)}

## Per-fold scores

In [ ]:
scores = {name: r.fold_scores() for name, r in results.items()}
summary = pd.concat({name: pd.concat([s, s.agg(["mean", "std"])]) for name, s in scores.items()})
diff = (scores["coarse dropout"] - scores["exp11"]).dropna()
diff = pd.concat([diff, diff.agg(["mean"])])
display(summary.round(3), diff.round(3))

## Macro F1 on all out-of-fold rows, thresholds from other folds

In [ ]:
def cross_f1(result):
    """Per-class threshold (tuned on all rows) and F1 with each fold scored by thresholds tuned on rest."""
    table = pd.DataFrame({"threshold": best_f1(result.labels, result.probs, CLASSES)["threshold"], "f1": cross_fold_f1(result.labels, result.probs, result.oof["fold"], CLASSES)})
    table.loc["Macro F1"] = [float("nan"), table["f1"].mean()]
    return table


pooled = pd.concat({name: cross_f1(r) for name, r in results.items()}, axis=1)
pooled.round(3)

## Confusion counts, VIP

In [ ]:
# best-F1 threshold per model and class on pooled OOF rows, as in submission
rows = []
for name, r in results.items():
    thresholds = best_f1(r.labels, r.probs, CLASSES)["threshold"]
    for cls in VIP:
        y, pred = r.oof[cls].eq(1), r.oof[f"p_{cls}"] >= thresholds[cls]
        rows.append({
            "model": name, "class": cls,
            "TP": int((y & pred).sum()), "FP": int((~y & pred).sum()),
            "FN": int((y & ~pred).sum()),
        })
confusion = pd.DataFrame(rows).set_index(["class", "model"]).sort_index(level=0, sort_remaining=False)
confusion

## Precision-recall curves

In [ ]:
fig = pr_curves(results, colors=SERIES, title="Out-of-fold rows of folds 0–4")
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

## Grad-CAM Cat, Dog: exp11 vs coarse dropout

In [ ]:
cams = {"exp11": GradCamFigures(EXP11), "coarse dropout": GradCamFigures(CONFIG)}
source = GradCamFigures(EXP16)
for cls in ["Cat", "Dog"]:
    rows = source.positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Grad-CAM VIP: unseen images

In [ ]:
for cls in VIP:
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_validation.png"] = cams["coarse dropout"].positives(cls, n=N_POSITIVES)
plt.show()

In [ ]:
with start_run(read_config(CONFIG)["experiment"], "summary", {"val_mode": "full"}):
    mlflow.log_table(summary.reset_index(names=["model", "fold"]), "fold_scores.json")
    mlflow.log_table(diff.reset_index(names="fold"), "fold_diff.json")
    flat = pooled.copy()
    flat.columns = [" ".join(c) for c in flat.columns]
    mlflow.log_table(flat.reset_index(names="class"), "pooled_f1.json")

save_figures(figures, FIGURES)